[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/27_deep_learning_foundations/05_tensorflow_and_keras.ipynb)

# Bài 05 — TensorFlow và Keras

**🎯 Mục tiêu**
- Đọc và viết được code **TensorFlow/Keras** — framework deep learning phổ biến thứ hai, gặp nhiều trong tutorial, sách và hệ thống triển khai (TF Serving, TF Lite, TF.js).
- `tf.Tensor`/`tf.Variable`, tự động tính đạo hàm với `tf.GradientTape`, `tf.function`, pipeline `tf.data`.
- Keras: Sequential và Functional API, `compile` / `fit` / `evaluate`, callback `EarlyStopping`, lưu/nạp mô hình.
- Vòng lặp huấn luyện tự viết bằng `GradientTape` — đối chiếu từng bước với PyTorch ([bài 04](04_pytorch_fundamentals.ipynb)).

> Mục *"TensorFlow"* (tuần 7) trong kế hoạch của khoá AIVN chưa có tài liệu — đây là bài soạn mới. Khoá học dùng **PyTorch** làm framework chính; bài này cho thấy cùng các khái niệm (tensor, autodiff, mô hình, loss, optimizer, vòng lặp huấn luyện) trong TensorFlow — khác cú pháp, giống tư duy.
>
> Cần TensorFlow: `pip install -e ".[tf]"` (Google Colab có sẵn). Trên Windows, TensorFlow ≥ 2.11 chỉ chạy CPU (GPU cần WSL2) — đủ cho bài này. Dữ liệu: Fashion-MNIST (tải ~30 MB lần đầu, dùng chung với assignment 07 · 93).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/27_deep_learning_foundations

In [ ]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")    # ẩn log C++ của TensorFlow
os.environ["KERAS_BACKEND"] = "tensorflow"            # Keras 3 chạy được trên nhiều backend — bài này dùng TensorFlow

import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from ml_course.real_datasets import FASHION_MNIST_CLASSES, load_fashion_mnist
from ml_course.visualization import set_style

try:
    import keras
    import tensorflow as tf
except ImportError as e:
    raise ImportError("Bài này cần TensorFlow: pip install -e \".[tf]\"") from e

tf.get_logger().setLevel("ERROR")
set_style()
keras.utils.set_random_seed(0)                         # seed cho Python, NumPy và TensorFlow cùng lúc
print(f"TensorFlow {tf.__version__} | Keras {keras.__version__} (backend: {keras.backend.backend()}) | "
      f"GPU: {len(tf.config.list_physical_devices('GPU'))}")

## 1. Tensor và Variable

`tf.Tensor` giống `ndarray`/`torch.Tensor`: có `shape`, `dtype`, broadcasting, chuyển qua lại với NumPy. Khác biệt quan trọng: **tensor của TensorFlow là bất biến** (không gán phần tử được); tham số cần cập nhật được đặt trong **`tf.Variable`**.

In [ ]:
x = tf.constant([[1.0, 2.0], [3.0, 4.0]])
print("dtype:", x.dtype.name, "| shape:", x.shape)
print("x @ xᵀ =\n", (x @ tf.transpose(x)).numpy())
print("broadcasting:", (x + tf.constant([10.0, 20.0])).numpy().tolist())
print("từ NumPy (float64 giữ nguyên!):", tf.constant(np.array([1.0, 2.0])).dtype.name)

w = tf.Variable([1.0, -1.0])
w.assign_sub([0.1, 0.1])                               # cập nhật tại chỗ — chỉ Variable làm được
print("Variable sau assign_sub:", w.numpy())
try:
    x[0, 0] = 5.0
except TypeError as e:
    print("Tensor bất biến →", type(e).__name__)

> ⚠️ NumPy mặc định `float64`, còn mạng nơ-ron (Keras, PyTorch) dùng `float32` — luôn ép `astype("float32")` khi đưa dữ liệu vào.

## 2. Tự động tính đạo hàm với `tf.GradientTape`

Mọi phép toán thực hiện **bên trong** `with tf.GradientTape() as tape:` được "ghi băng"; `tape.gradient(loss, [biến…])` chạy lan truyền ngược trên băng đó. So với PyTorch:

| PyTorch | TensorFlow |
|---|---|
| `loss.backward()` rồi đọc `w.grad` | `grads = tape.gradient(loss, [w, b])` — trả về trực tiếp |
| gradient **cộng dồn** → cần `zero_grad()` | không cộng dồn — mỗi băng dùng một lần |
| mọi tensor `requires_grad=True` | tự theo dõi `tf.Variable`; tensor thường cần `tape.watch(x)` |

Hồi quy tuyến tính "bằng tay" — giống mục 3 của bài 04:

In [ ]:
rng = np.random.default_rng(0)
x_data = rng.uniform(-1, 1, 200).astype("float32")
y_data = (3 * x_data + 2 + rng.normal(0, 0.1, 200)).astype("float32")

w, b = tf.Variable(0.0), tf.Variable(0.0)
for step in range(200):
    with tf.GradientTape() as tape:
        loss = tf.reduce_mean((w * x_data + b - y_data) ** 2)
    dw, db = tape.gradient(loss, [w, b])
    w.assign_sub(0.1 * dw)
    b.assign_sub(0.1 * db)
print(f"w = {w.numpy():.3f} (thật: 3) | b = {b.numpy():.3f} (thật: 2) | loss = {loss.numpy():.4f}")

## 3. `tf.function`: từ Python sang đồ thị tính toán

Mặc định TensorFlow chạy **eager** (từng lệnh một, như PyTorch). Bọc hàm bằng `@tf.function`, lần gọi đầu TensorFlow **truy vết** (trace) hàm thành một đồ thị, các lần sau chạy đồ thị đó — bỏ qua chi phí của Python, tối ưu và gộp phép toán.

In [ ]:
def many_small_ops(x):
    for _ in range(10):
        x = tf.tanh(x @ tf.transpose(x) * 0.01)
    return x


graph_version = tf.function(many_small_ops)
x_in = tf.random.normal((64, 64))
graph_version(x_in)                                    # lần đầu: truy vết
for name, fn in [("eager", many_small_ops), ("tf.function", graph_version)]:
    t0 = time.perf_counter()
    for _ in range(300):
        fn(x_in)
    print(f"{name:12s}: {(time.perf_counter() - t0) / 300 * 1e3:.2f} ms mỗi lần gọi")

Bẫy hay gặp: code **Python** bên trong `tf.function` (như `print`, thêm vào list) chỉ chạy **lúc truy vết**, không chạy mỗi lần gọi. Dùng `tf.print` cho thứ cần in mỗi lần:

In [ ]:
@tf.function
def traced(x):
    print("print của Python — chỉ chạy lúc truy vết")
    tf.print("tf.print — chạy mỗi lần gọi, x =", x)
    return x * 2


for v in [1.0, 2.0, 3.0]:
    traced(tf.constant(v))

## 4. `tf.data`: pipeline dữ liệu

`tf.data.Dataset` tương đương `Dataset` + `DataLoader` của PyTorch: xáo trộn, chia batch, và `prefetch` để chuẩn bị batch kế tiếp trong lúc mô hình đang tính.

In [ ]:
X_train_img, y_train, X_test_img, y_test = load_fashion_mnist()
X_train = X_train_img.reshape(len(X_train_img), -1).astype("float32") / 255.0
X_test = X_test_img.reshape(len(X_test_img), -1).astype("float32") / 255.0

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(len(X_train), seed=0)
            .batch(128)
            .prefetch(tf.data.AUTOTUNE))
xb, yb = next(iter(train_ds))
print("một batch:", xb.shape, xb.dtype.name, "| nhãn:", yb.shape, "| số batch mỗi epoch:", len(train_ds))

## 5. Keras: mô hình trong vài dòng

**Keras** là API cấp cao của TensorFlow: khai báo mô hình, `compile` (optimizer + loss + metric), `fit`, `evaluate`, `predict`. Một MLP 784 → 256 → 128 → 10 cho Fashion-MNIST:

In [ ]:
def make_mlp():
    return keras.Sequential([
        keras.Input(shape=(784,)),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dropout(0.2),                     # chương 28
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10),                        # logits — KHÔNG softmax ở đây
    ])


keras.utils.set_random_seed(0)
model = make_mlp()
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),   # nhãn là số nguyên 0..9
              metrics=["accuracy"])
model.summary()

`Dense(256)` tự suy ra số đầu vào từ lớp trước (PyTorch phải ghi rõ `nn.Linear(784, 256)`). Số tham số lớp đầu: $784 \cdot 256 + 256 = 200\,960$.

`fit` lo toàn bộ vòng lặp huấn luyện. `validation_split=0.1` giữ lại 10% cuối của tập train để theo dõi; callback **EarlyStopping** dừng khi `val_loss` không cải thiện sau 3 epoch và khôi phục trọng số tốt nhất (chương 28):

In [ ]:
t0 = time.perf_counter()
history = model.fit(X_train, y_train, validation_split=0.1, epochs=20, batch_size=128, verbose=0,
                    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)])
n_epochs = len(history.history["loss"])
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"dừng sau {n_epochs} epoch ({time.perf_counter() - t0:.0f}s) | epoch tốt nhất theo val_loss: "
      f"{int(np.argmin(history.history['val_loss'])) + 1} | accuracy test = {test_acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for key, ax in zip(["loss", "accuracy"], axes):
    ax.plot(history.history[key], "o-", ms=3, label="train")
    ax.plot(history.history[f"val_{key}"], "s-", ms=3, label="validation")
    ax.set_xlabel("epoch")
    ax.set_title(key)
    ax.legend()
plt.show()

MLP hai lớp ẩn đạt khoảng **88%** trên tập test — so với ~83% của softmax regression (mô hình tuyến tính) trong [assignment 07 · 93](../07_logistic_regression/93_assignment_softmax_regression.ipynb). Lớp ẩn học được đặc trưng phi tuyến; CNN ([chương 29](../29_cnn/README.md)) còn khai thác cấu trúc không gian của ảnh để đi xa hơn.

Dự đoán: `predict` trả về **logits** (vì lớp cuối không có softmax) — muốn xác suất thì áp `softmax`:

In [ ]:
logits = model.predict(X_test[:5], verbose=0)
proba = keras.ops.softmax(logits).numpy()
for k in range(5):
    print(f"thật: {FASHION_MNIST_CLASSES[y_test[k]]:>13s} | dự đoán: {FASHION_MNIST_CLASSES[proba[k].argmax()]:>13s} "
          f"(xác suất {proba[k].max():.2f})")

## 6. Functional API và subclassing

`Sequential` chỉ cho mô hình **một chuỗi** lớp. **Functional API** mô tả mô hình như một đồ thị: nhiều đầu vào/đầu ra, nhánh, kết nối tắt (residual — chương 29). Cùng MLP ở trên, cộng thêm một kết nối tắt:

In [ ]:
inputs = keras.Input(shape=(784,))
h1 = keras.layers.Dense(128, activation="relu")(inputs)
h2 = keras.layers.Dense(128, activation="relu")(h1)
merged = keras.layers.Add()([h1, h2])                  # kết nối tắt: h1 + h2
outputs = keras.layers.Dense(10)(merged)
functional_model = keras.Model(inputs, outputs, name="mlp_residual")
print("số tham số:", functional_model.count_params())
print([layer.__class__.__name__ for layer in functional_model.layers])

Khi cần logic tuỳ ý trong lúc tính (vòng lặp, điều kiện), kế thừa `keras.Model` và viết `call()` — tương đương kế thừa `nn.Module` và viết `forward()` trong PyTorch.

## 7. Vòng lặp huấn luyện tự viết với `GradientTape`

`fit` tiện, nhưng các mô hình đặc biệt (GAN, học tăng cường, loss tuỳ biến) cần vòng lặp riêng. Đối chiếu từng bước với PyTorch:

| Bước | PyTorch | TensorFlow |
|---|---|---|
| 1. forward | `logits = model(xb)` | `with tf.GradientTape() as tape: logits = model(xb, training=True)` |
| 2. loss | `loss = loss_fn(logits, yb)` | `loss = loss_fn(yb, logits)` (chú ý **thứ tự** tham số!) |
| 3. xoá gradient cũ | `optimizer.zero_grad()` | không cần |
| 4. backward | `loss.backward()` | `grads = tape.gradient(loss, model.trainable_variables)` |
| 5. cập nhật | `optimizer.step()` | `optimizer.apply_gradients(zip(grads, model.trainable_variables))` |

In [ ]:
keras.utils.set_random_seed(0)
custom_model = make_mlp()
optimizer = keras.optimizers.Adam(learning_rate=1e-3)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)


@tf.function                                           # biên dịch bước huấn luyện thành đồ thị
def train_step(xb, yb):
    with tf.GradientTape() as tape:
        loss = loss_fn(yb, custom_model(xb, training=True))      # training=True bật dropout
    grads = tape.gradient(loss, custom_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, custom_model.trainable_variables))
    return loss


t0 = time.perf_counter()
for epoch in range(5):
    epoch_loss = np.mean([float(train_step(xb, yb)) for xb, yb in train_ds])
    acc = np.mean(np.argmax(custom_model.predict(X_test, verbose=0), axis=1) == y_test)
    print(f"epoch {epoch + 1}: loss train = {epoch_loss:.4f} | accuracy test = {acc:.4f}")
print(f"({time.perf_counter() - t0:.0f}s)")

Cùng kiến trúc, 5 epoch vòng lặp tự viết cho kết quả tương đương `fit` — `fit` chỉ là vòng lặp này cộng thêm validation, callback và thanh tiến trình.

## 8. Lưu và nạp mô hình

Định dạng `.keras` lưu **kiến trúc + trọng số + trạng thái optimizer** trong một file. (Triển khai thường dùng định dạng SavedModel/`model.export(...)` cho TF Serving, hoặc chuyển sang TF Lite cho điện thoại.)

In [ ]:
path = Path(tempfile.mkdtemp()) / "fashion_mlp.keras"
model.save(path)
restored = keras.models.load_model(path)
same = np.allclose(model.predict(X_test[:100], verbose=0), restored.predict(X_test[:100], verbose=0), atol=1e-5)
print(f"đã lưu {path.name} ({path.stat().st_size / 1e6:.1f} MB) | dự đoán sau khi nạp giống hệt: {same}")

## 9. PyTorch ↔ TensorFlow/Keras: bảng tương đương

| Khái niệm | PyTorch | TensorFlow / Keras |
|---|---|---|
| Tensor | `torch.tensor(...)` (khả biến) | `tf.constant(...)` (bất biến) |
| Tham số | tensor `requires_grad=True`, `nn.Parameter` | `tf.Variable` |
| Đạo hàm | `loss.backward()`, `.grad` | `tf.GradientTape`, `tape.gradient` |
| Mô hình | `nn.Module` + `forward()` | `keras.Sequential`, Functional API, `keras.Model` + `call()` |
| Lớp tuyến tính | `nn.Linear(in, out)` | `keras.layers.Dense(out)` (tự suy `in`) |
| Loss phân loại | `nn.CrossEntropyLoss()` (nhận logits) | `SparseCategoricalCrossentropy(from_logits=True)` |
| Dữ liệu | `Dataset` + `DataLoader` | `tf.data.Dataset` |
| Huấn luyện | vòng lặp tự viết | `model.fit` (hoặc vòng lặp tự viết) |
| Chế độ train/eval | `model.train()` / `model.eval()` | tham số `training=True/False` (`fit`/`predict` tự đặt) |
| Lưu mô hình | `torch.save(model.state_dict())` | `model.save("m.keras")` |
| Ảnh (thứ tự chiều) | `(N, C, H, W)` | `(N, H, W, C)` |
| Triển khai | TorchScript, ONNX, TorchServe | SavedModel, TF Serving, TF Lite, TF.js |

**Chọn framework nào?** Cả hai đều đủ mạnh cho mọi bài toán trong khoá học. PyTorch chiếm đa số trong nghiên cứu và mã nguồn của các bài báo mới; TensorFlow/Keras mạnh ở hệ sinh thái triển khai (điện thoại, trình duyệt, TF Serving) và có nhiều hệ thống production cũ. Keras 3 còn chạy được trên **nhiều backend** (TensorFlow, JAX, PyTorch — đổi bằng biến môi trường `KERAS_BACKEND`). Hiểu khái niệm quan trọng hơn thuộc cú pháp: chuyển qua lại chỉ mất vài ngày.

## 10. Lỗi thường gặp

| Triệu chứng | Nguyên nhân thường gặp |
|---|---|
| Loss cao bất thường, học rất chậm | lớp cuối đã có `softmax` **và** loss đặt `from_logits=True` (hoặc ngược lại: lớp cuối ra logits mà quên `from_logits=True`) |
| Lỗi shape giữa nhãn và đầu ra | nhãn số nguyên phải dùng `SparseCategoricalCrossentropy`; nhãn one-hot dùng `CategoricalCrossentropy` |
| `print` trong `tf.function` chỉ in một lần | code Python chạy lúc truy vết → dùng `tf.print` |
| `tape.gradient` trả về `None` | tính toán đi qua NumPy (ngắt băng ghi), hoặc biến là `tf.constant` chưa `tape.watch` |
| Chạy chậm, cảnh báo *retracing* | gọi `tf.function` với số Python/shape thay đổi liên tục → truyền tensor, cố định shape |
| Lỗi kiểu `float64` vs `float32` | dữ liệu NumPy chưa `astype("float32")` |

## 🧠 Kiểm tra nhanh

1. Vì sao TensorFlow tách `tf.Variable` khỏi `tf.Tensor`? PyTorch xử lý tham số thế nào?
2. Trong vòng lặp tự viết bằng TensorFlow, vì sao không cần bước tương đương `optimizer.zero_grad()`?
3. Một hàm `@tf.function` có `print("hello")` được gọi 100 lần. "hello" được in mấy lần? Vì sao?
4. Lớp cuối `Dense(10, activation="softmax")` đi với loss nào cho đúng?
5. `model.fit(..., validation_split=0.1)` lấy những mẫu nào làm validation? Điều gì xảy ra nếu dữ liệu đã được sắp xếp theo nhãn?

## 📌 Tóm tắt

- `tf.Tensor` bất biến, `tf.Variable` cho tham số; `tf.GradientTape` ghi phép toán và tính gradient trực tiếp (không cộng dồn).
- `tf.function` biến hàm Python thành đồ thị — nhanh hơn, nhưng code Python chỉ chạy lúc truy vết.
- Keras: `Sequential`/Functional → `compile` → `fit` (callback EarlyStopping) → `evaluate`/`predict` → `save(".keras")`.
- Vòng lặp tự viết: forward trong `GradientTape` → loss → `tape.gradient` → `apply_gradients`.
- MLP trên Fashion-MNIST ~88% so với ~83% của mô hình tuyến tính — cùng dữ liệu, thêm lớp ẩn phi tuyến.

## 📚 Tài liệu tham khảo

- TensorFlow Guide — [tensorflow.org/guide](https://www.tensorflow.org/guide) (Tensors, Variables, Autodiff, `tf.function`, `tf.data`).
- Keras — [keras.io/guides](https://keras.io/guides/) (Sequential, Functional API, writing a training loop from scratch).
- F. Chollet — *Deep Learning with Python* (Manning).
- A. Géron — *Hands-On Machine Learning with Scikit-Learn, Keras and TensorFlow* (O'Reilly).

<sub>Chủ đề *"TensorFlow"* (tuần 7) trong kế hoạch khoá [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning) — bản gốc chưa có tài liệu; nội dung soạn mới cho khoá học này, đối chiếu với PyTorch.</sub>